<a id="top"></a>

# Non-determinism, pass rates, and a model A/B

```yaml
title: "Non-determinism, pass rates, and a model A/B"
keywords: evaluation, non-determinism, pass rate, samples, model comparison, sonnet, haiku, regression, measurement instrument, Langfuse, experiment, run comparison, run_experiment, teacher lecture
estimated duration: 30
```

> **Lesson:** L13 Evaluation. Roadmap: [objectives.md](../../../../docs/origin/lesson_roadmaps/L13/objectives.md) (objective 3).
>
> Sections 1–4 run **offline** — no API key — with the model contrast **simulated deterministically** so the page reproduces. Section 4.1 then runs the *real* Sonnet-vs-Haiku A/B as **two Langfuse experiments over the shared `l13-agent-evals` dataset**; that tooled cell soft-skips unless *both* a model key and the cohort Langfuse are configured, so a keyless restart-and-run-all still finishes.

## Contents

- [1. Setup — two simulated models](#1-setup--two-simulated-models)
- [2. One run can be luck](#2-one-run-can-be-luck)
- [3. Measure rates, not verdicts](#3-measure-rates-not-verdicts)
- [4. Same eval set, two models](#4-same-eval-set-two-models)
  - [4.1 (Live) the real A/B as two Langfuse experiments](#41-live-the-real-ab-as-two-langfuse-experiments)
  - [4.2 Reading the run-comparison view](#42-reading-the-run-comparison-view)
- [5. The same machinery is a regression check](#5-the-same-machinery-is-a-regression-check)
- [6. Takeaways](#6-takeaways)

## 1. Setup — two simulated models

L12 told you the graph is **non-deterministic**: the same task can take a different path each run, so *a single differing run proves nothing*. L13 builds the disciplined version of that warning.

To show it without a live model (and without flakiness we can't reproduce on the page), we **simulate** two models with a scripted `FakeModel`:

- a **strong** model that chains correctly and recovers from a tool error;
- a **weak** model that gives up on recovery and *intermittently runs away* on the lookup task.

`ModelSim` is keyed by `(case.id, attempt#)`, so it's flaky in a reproducible way. The two scorers — `answer_correct` (loosened to ignore commas) and `no_runaway` — apply to every case.

In [ ]:
from typing import Any

from fluffy_potato_curriculum.common.agent_graph import RunResult, arun
from fluffy_potato_curriculum.common.evals import (
    EvalCase,
    EvalResult,
    Scorer,
    aevaluate,
    compare,
    tool_calls,
)
from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)
from fluffy_potato_curriculum.common.tools import TOOLS


def _norm(text: str) -> str:
    """Drop commas/spaces so '37,000,000' and '37000000' compare equal."""
    return text.replace(",", "").replace(" ", "")


def answer_correct(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Outcome scorer (loosened): is the reference answer in the final text?"""
    expected = str(example.reference_outputs["answer"])
    return EvalResult(key="answer_correct", score=_norm(expected) in _norm(run.final_text))


def no_runaway(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Trajectory scorer: no repeated (tool, args) call and didn't hit the step cap."""
    signatures = [(name, tuple(sorted(args.items()))) for name, args in tool_calls(run)]
    repeated = len(signatures) != len(set(signatures))
    return EvalResult(key="no_runaway", score=not (repeated or run.termination == "max_steps"))


class ModelSim:
    """A run_case that simulates a model of a given strength, deterministically.

    Strong: recovers from the crash (retries https://ok) and looks Tokyo up once.
    Weak: gives up on recovery, and on every third lookup attempt runs away on a
    bad city until max_steps. Behavior is keyed by (case.id, attempt#).
    """

    def __init__(self, *, strong: bool) -> None:
        self.strong = strong
        self._seen: dict[str, int] = {}

    def _attempt(self, case_id: str) -> int:
        n = self._seen.get(case_id, 0)
        self._seen[case_id] = n + 1
        return n

    async def __call__(self, case: EvalCase) -> RunResult:
        n = self._attempt(case.id)
        if case.id == "recover" and self.strong:
            script = [
                tool_reply(tool_call("c1", "flaky_fetch", {"url": "https://crash"})),
                tool_reply(tool_call("c2", "flaky_fetch", {"url": "https://ok"})),
                text_reply("The answer is 42."),
            ]
        elif case.id == "recover":
            script = [
                tool_reply(tool_call("c1", "flaky_fetch", {"url": "https://crash"})),
                text_reply("Sorry, I could not fetch that URL."),
            ]
        elif case.id == "hard_lookup" and not self.strong and n % 3 == 1:
            # the weak model flakes into a runaway on every third attempt. Distinct
            # call ids (r0, r1, ...) keep the graph's add_messages reducer from
            # de-duping the repeats, so it genuinely loops to the max_steps cap.
            script = [
                tool_reply(tool_call(f"r{i}", "lookup", {"city": "Atlantis"})) for i in range(8)
            ]
        elif case.id == "hard_lookup":
            script = [
                tool_reply(tool_call("c1", "lookup", {"city": "Tokyo"})),
                text_reply("Tokyo has a population of 37000000."),
            ]
        else:
            raise KeyError(case.id)
        return await arun(
            FakeModel(script), TOOLS, case.inputs["task"], max_steps=case.inputs.get("max_steps", 8)
        )


recover_case = EvalCase(
    id="recover",
    inputs={"task": "Fetch the answer from https://crash; if it fails, try https://ok."},
    reference_outputs={"answer": "42"},
)
hard_lookup_case = EvalCase(
    id="hard_lookup",
    inputs={"task": "What is the population of Tokyo?", "max_steps": 4},
    reference_outputs={"answer": "37000000"},
)
ab_cases = [recover_case, hard_lookup_case]
scorers = [answer_correct, no_runaway]
print("setup ready:", [c.id for c in ab_cases])

[↑ Back to top](#top)

## 2. One run can be luck

Run the weak model on the lookup case **once** (`samples=1`). It passes — looks fine. Run it again. It fails. *Which run do we believe?*

This confusion **is** the lesson. A single pass on a non-deterministic agent can be luck (and a single fail can be an unlucky path).

In [ ]:
weak_once = ModelSim(strong=False)

print("first run (samples=1):")
print((await aevaluate(weak_once, [hard_lookup_case], scorers)).table())

print("\nsecond run (samples=1):")
print((await aevaluate(weak_once, [hard_lookup_case], scorers)).table())

First run green, second run red — same case, same model, no code changed. Believing either single run is believing a coin flip.

[↑ Back to top](#top)

## 3. Measure rates, not verdicts

The cheapest honest fix: run each case **K times** and report how often it passes — a **pass rate**, not a single verdict. `aevaluate(..., samples=K)` does exactly this. With `K=3` the flaky case reads as `2/3`: a *finding*, not noise to ignore. (A flaky case needs a looser check, more samples, or a redesign — not a shrug.)

In [ ]:
weak = ModelSim(strong=False)
report_k = await aevaluate(weak, [hard_lookup_case], scorers, samples=3)
print(report_k.table())
print("\nno_runaway pass rate:", report_k.pass_rate("hard_lookup", "no_runaway"))

[↑ Back to top](#top)

## 4. Same eval set, two models

Now the headline. Run the **same** eval set against the strong model and the weak model, `K=3` samples each, and put the pass-rate tables side by side. Watch the weak model's rates **drop** — especially on the recovery case and the flaky multi-step lookup.

This turns "the weaker model is worse here" from an assertion into a **number**. The eval set is not just a regression guard; it's a **measurement instrument**.

In [ ]:
strong_report = await aevaluate(ModelSim(strong=True), ab_cases, scorers, samples=3)
weak_report = await aevaluate(ModelSim(strong=False), ab_cases, scorers, samples=3)

print("=== strong model (Sonnet stand-in) ===")
print(strong_report.table())
print("\n=== weak model (Haiku stand-in) ===")
print(weak_report.table())

The strong model is green across the board; the weak model drops to `0/3` on recovery (it gives up instead of retrying) and `2/3` on the lookup (it intermittently runs away). A green eval set on the strong model would have told you *nothing* about this headroom — you only see it by running the *same cases* against both.

### 4.1 (Live) the real A/B as two Langfuse experiments

Now do the A/B the way you actually would: run the **shared `l13-agent-evals` dataset** (uploaded in `L1302_lecture`) as **two Langfuse experiments** — one driving **Sonnet 4.6**, one **Haiku 4.5** — through the *same* graph and the *same* two checks.

`dataset.run_experiment(name=..., task=..., evaluators=[...])` is Langfuse's batteries-included runner: it runs your `task` on every dataset item, applies each `evaluator`, and records the whole thing as one **dataset run** you can open in the UI. (It's a sync SDK method with no async twin, but it happily accepts an `async def` task and awaits it per item — so our async graph drops straight in.) Two reuse beats worth naming:

- the **task** is just our `agent_graph.arun(...)` — it returns a `RunResult`;
- each **evaluator** is one of the *same* scorers from Sections 2–4, wrapped by a one-line `as_evaluator(...)` adapter. A Langfuse evaluator is called `(*, input, output, expected_output, ...)` and returns `{"name", "value", "comment"}` — so **your scorer *is* the evaluator**.

The cell is gated on **both** `ANTHROPIC_API_KEY` (the live models) and `langfuse_configured()` (the experiment store); with either unset it prints a note and skips, so a keyless restart-and-run-all still passes. We launch `SAMPLES` runs per model — the pass **rate** across those runs is the honest read, exactly as in Section 3.

In [ ]:
# The live A/B, done the way you actually would: run the SHARED `l13-agent-evals`
# dataset as two Langfuse EXPERIMENTS (Sonnet vs Haiku). `dataset.run_experiment`
# runs `task` on every item, applies the `evaluators`, and records one dataset RUN.
# Gated on BOTH a model key and a live Langfuse, so a keyless run still finishes.
from fluffy_potato_curriculum.common.config import (
    get_settings,
    langfuse_configured,
    require_langfuse,
)

DATASET_NAME = "l13-agent-evals"  # uploaded in L1302 (and by the L1303 lab)
SAMPLES = 3  # re-run each model K times; the pass RATE across runs is the honest read
MODELS = [
    ("sonnet-4-6", "anthropic:claude-sonnet-4-6"),
    ("haiku-4-5", "anthropic:claude-haiku-4-5-20251001"),
]


def as_evaluator(scorer: Scorer):
    """Adapt a course `Scorer` into a Langfuse evaluator — your scorer *is* the evaluator.

    A Langfuse evaluator is called `(*, input, output, expected_output, ...)` and returns
    `{"name", "value", "comment"}`. Our `output` is the `RunResult` the task returned, so
    the scorer runs unchanged. Return `[]` to skip an item the scorer can't judge (e.g.
    `answer_correct` on a runaway case that carries no reference answer).
    """

    def evaluator(
        *,
        input: dict[str, Any],
        output: RunResult,
        expected_output: dict[str, Any] | None = None,
        **kwargs: Any,
    ) -> dict[str, Any] | list[dict[str, Any]]:
        example = EvalCase(id="", inputs=input, reference_outputs=expected_output or {})
        try:
            verdict = scorer(run=output, example=example)
        except KeyError:
            return []  # no reference for this scorer on this item -> don't score it
        return {"name": verdict.key, "value": verdict.score, "comment": verdict.comment}

    return evaluator


def make_task(model_id: str):
    """A Langfuse task: drive our graph with a real model on one dataset item.

    The task is `async def` — `run_experiment` (itself sync, no async twin)
    detects and awaits an awaitable task per item.
    """
    from langchain.chat_models import init_chat_model

    # keys load through the config seam (a repo-root .env), so pass explicitly.
    model = init_chat_model(model_id, api_key=get_settings().anthropic_api_key)

    async def task(*, item: Any, **kwargs: Any) -> RunResult:
        return await arun(
            model, TOOLS, item.input["task"], max_steps=item.input.get("max_steps", 8)
        )

    return task


if langfuse_configured() and get_settings().anthropic_api_key:
    from langfuse import Langfuse

    host, public_key, secret_key = require_langfuse()
    client = Langfuse(host=host, public_key=public_key, secret_key=secret_key)
    dataset = client.get_dataset(DATASET_NAME)  # uploaded by L1302
    evaluators = [as_evaluator(answer_correct), as_evaluator(no_runaway)]

    for label, model_id in MODELS:
        run_urls: list[str | None] = []
        for k in range(SAMPLES):
            result = dataset.run_experiment(
                name="L13 model A/B",
                run_name=f"{label} · sample {k + 1}",
                task=make_task(model_id),
                evaluators=evaluators,
            )
            run_urls.append(result.dataset_run_url)
        print(f"{label}: launched {len(run_urls)} runs — compare them in Langfuse: {run_urls[-1]}")
    client.flush()
else:
    print("Langfuse and/or ANTHROPIC_API_KEY not set — skipping the live experiment A/B.")
    print("The offline strong-vs-weak comparison above already made the point; with both")
    print("configured, this runs the SAME l13-agent-evals dataset as two Langfuse experiments.")

### 4.2 Reading the run-comparison view

Open the **`l13-agent-evals`** dataset in Langfuse and switch to its **Runs** tab. Each experiment you launched is a row; select the Sonnet runs and the Haiku runs to line them up **item by item**. The weaker model's `answer_correct` / `no_runaway` averages drop on exactly the cases the offline tables predicted — the recovery and runaway items — only now it's real models over your real dataset, comparable at a click.

This same view is your **regression** view: re-run the experiment after a prompt edit and compare the new run against the old one. "Did model B beat model A?" and "did my prompt edit break a case that used to pass?" are the *same* comparison — which is exactly what the offline `compare()` in the next section does by hand.

[↑ Back to top](#top)

## 5. The same machinery is a regression check

Section 4.2 read the regression off the Langfuse **run comparison**; here is the *same* check as ~15 lines of local Python, so the hosted view is never magic. Treat the strong model's report as the **baseline** and ask: which (case, scorer) pairs went **pass → fail**? `compare(before, after)` answers it — a pair "passes" when its rate hits `min_rate` (default `1.0`: green on every sample).

The same call answers "did my prompt edit break anything that used to work?" — *same machinery, different question.* Hosted (Langfuse runs) or local (`compare`), the ratchet is identical.

In [6]:
delta = compare(strong_report, weak_report)
print("regressions (pass -> fail):", delta.regressions)
print("fixes       (fail -> pass):", delta.fixes)
print("has regressions:", delta.has_regressions)

regressions (pass -> fail): [('recover', 'answer_correct'), ('hard_lookup', 'answer_correct'), ('hard_lookup', 'no_runaway')]
fixes       (fail -> pass): []
has regressions: True


Swapping in the weaker model regressed recovery and the lookup — caught *before* shipping, by re-running cases you already had. That is the **ratchet**: once a case passes, a later change that breaks it is a regression you catch, not one you rediscover in production.

[↑ Back to top](#top)

## 6. Takeaways

- **You measure rates, not verdicts.** One green run on a non-deterministic agent can be luck. A **pass rate** over a few samples is the cheapest answer you can trust — the disciplined version of L12's "a single differing run proves nothing."
- **A flaky case is a finding.** `2/3` isn't noise to ignore; it means the case needs a looser check, more samples, or a redesign.
- **The eval set is a measurement instrument.** Running the *same* cases against two models turns "this one is weaker" into a number — and exposes headroom a single green run hides.
- **The platform runs the A/B for you.** `dataset.run_experiment(task=..., evaluators=[...])` runs the shared `l13-agent-evals` dataset as one **dataset run** per model — your `agent_graph.arun` is the task (an async task it awaits per item) and your **scorers are the evaluators** (one adapter line). The runs line up in Langfuse's **run-comparison view**; it's a hard dependency here, gated on the model + Langfuse keys.
- **The ratchet is the payoff.** `compare(before, after)` — or the same Langfuse run comparison — flags pass→fail regressions before they ship. The same machinery answers "model A vs B" and "before vs after a prompt edit."
- **Next:** [`L1305_lab`](L1305_lab_empty.ipynb) — you run an eval set at `samples=K`, read pass rates, and flag regressions across two versions yourself, then push a run to Langfuse. Then [`L1306_lecture`](L1306_lecture.ipynb) asks what all this *costs*.

[↑ Back to top](#top)